<a href="https://colab.research.google.com/github/ishan-singh-1/hcl/blob/main/ishan%20singh(2026b03110207).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

PATH = "15_cinema_sales_1000.csv"

# ---- Column names (edit to match your file) ----
COL = dict(
    txn="Transaction_ID",
    movie="Movie_Title",
    genre="Genre",
    fmt="Screen_Format",
    date="Show_Date",
    slot="Showtime_Slot",
    qty="Tickets_Quantity",
    price="Ticket_Price",
    conc="Concession_Spend",
    occ="Occupancy_Percentage",
)
# If your file already has a ticket revenue column, put its name here (else None)
TICKET_REVENUE_COL = None


def header(t):
    print("\n" + "=" * 70 + f"\n{t}\n" + "=" * 70)


In [2]:
df = pd.read_csv(PATH)
header("1. DATA INSPECTION")
print("Shape:", df.shape)
print(df.dtypes)
print(df.head())
print(df.describe(include="all").T)

missing = [c for c in COL.values() if c not in df.columns]
if missing:
    raise KeyError(f"Columns not found in file: {missing}. "
                   f"Actual columns: {list(df.columns)}. Edit the COL dictionary.")



1. DATA INSPECTION
Shape: (1000, 10)
Transaction_ID           object
Show_Date                object
Movie_Title              object
Genre                    object
Showtime_Slot            object
Screen_Format            object
Tickets_Quantity          int64
Ticket_Price            float64
Occupancy_Percentage    float64
Concession_Spend        float64
dtype: object
  Transaction_ID   Show_Date       Movie_Title      Genre Showtime_Slot Screen_Format  Tickets_Quantity  Ticket_Price  Occupancy_Percentage  Concession_Spend
0         C00000  2026-10-13  The Last Journey  Animation    Late Night   Standard 2D                 4        154.21                  26.8            557.44
1         C00001  2025-07-05       Hidden City     Action    Late Night            3D                 6        182.11                  67.1               NaN
2         C00002  2025-05-02    Laugh Out Loud     Action    Late Night            3D                 2        287.64                  47.6            544

In [3]:
header("2. MISSING VALUES & DUPLICATES")
print(df.isna().sum())
dups = df[df.duplicated(COL["txn"], keep=False)].sort_values(COL["txn"])
print(f"Rows sharing a duplicate Transaction_ID: {len(dups)}")
if len(dups):
    print(dups.head(10))
before = len(df)
df = df.drop_duplicates(subset=COL["txn"], keep="first").copy()
print(f"Dropped {before - len(df)} duplicate rows (kept first occurrence).")


2. MISSING VALUES & DUPLICATES
Transaction_ID           0
Show_Date                0
Movie_Title              0
Genre                    0
Showtime_Slot            0
Screen_Format            0
Tickets_Quantity         0
Ticket_Price             0
Occupancy_Percentage     0
Concession_Spend        30
dtype: int64
Rows sharing a duplicate Transaction_ID: 20
    Transaction_ID   Show_Date     Movie_Title      Genre Showtime_Slot Screen_Format  Tickets_Quantity  Ticket_Price  Occupancy_Percentage  Concession_Spend
155         C00155  2025-07-25  Laugh Out Loud     Comedy       Evening          IMAX                 1        191.39                  55.1            141.99
703         C00155  2025-07-25  Laugh Out Loud     Comedy       Evening          IMAX                 1        191.39                  55.1            141.99
158         C00158  2025-12-22    Robot Reborn  Animation       Evening            3D                 6        345.88                  95.2            299.66
700      

In [4]:
header("3. CONCESSION_SPEND IMPUTATION")
df["Conc_Raw"] = df[COL["conc"]]          # keep original (pre-imputation) values
n_missing = df[COL["conc"]].isna().sum()
# Strategy: median of same genre + slot (spend is right-skewed, so median > mean),
# falling back to the overall median.
group_median = df.groupby([COL["genre"], COL["slot"]])[COL["conc"]].transform("median")
df[COL["conc"]] = (df[COL["conc"]]
                   .fillna(group_median)
                   .fillna(df[COL["conc"]].median()))
print(f"Imputed {n_missing} missing values with genre x slot median.")
print("Remaining missing:", df[COL["conc"]].isna().sum())


3. CONCESSION_SPEND IMPUTATION
Imputed 30 missing values with genre x slot median.
Remaining missing: 0


In [5]:
header("4. DATE FEATURES")
df[COL["date"]] = pd.to_datetime(df[COL["date"]], errors="coerce")
print("Unparseable dates:", df[COL["date"]].isna().sum())
df["Weekday"] = df[COL["date"]].dt.day_name()
df["Day_Type"] = np.where(df["Weekday"].isin(["Saturday", "Sunday"]), "Weekend", "Weekday")
print(df[[COL["date"], "Weekday", "Day_Type"]].head())


4. DATE FEATURES
Unparseable dates: 0
   Show_Date   Weekday Day_Type
0 2026-10-13   Tuesday  Weekday
1 2025-07-05  Saturday  Weekend
2 2025-05-02    Friday  Weekday
3 2025-08-05   Tuesday  Weekday
4 2025-04-27    Sunday  Weekend


In [6]:
header("5. REVENUE TOTALS")
if TICKET_REVENUE_COL:
    df["Ticket_Revenue"] = df[TICKET_REVENUE_COL]
else:
    df["Ticket_Revenue"] = df[COL["qty"]] * df[COL["price"]]
df["Total_Spend"] = df["Ticket_Revenue"] + df[COL["conc"]]
tix_rev = df["Ticket_Revenue"].sum()
con_rev = df[COL["conc"]].sum()
print(f"Total ticket revenue     : {tix_rev:,.2f}")
print(f"Total concession revenue : {con_rev:,.2f}")
print(f"Combined revenue         : {tix_rev + con_rev:,.2f}")
print(f"Concession share of total: {con_rev / (tix_rev + con_rev):.1%}")


5. REVENUE TOTALS
Total ticket revenue     : 1,233,486.32
Total concession revenue : 337,266.52
Combined revenue         : 1,570,752.83
Concession share of total: 21.5%


In [7]:
header("6. TOP 10 MOVIES")
movies = (df.groupby(COL["movie"])
            .agg(Tickets_Sold=(COL["qty"], "sum"),
                 Ticket_Revenue=("Ticket_Revenue", "sum"))
            .reset_index())
top_by_tickets = movies.sort_values("Tickets_Sold", ascending=False).head(10)
top_by_revenue = movies.sort_values("Ticket_Revenue", ascending=False).head(10)
print("By tickets sold:\n", top_by_tickets.to_string(index=False))
print("\nBy ticket revenue:\n", top_by_revenue.to_string(index=False))


6. TOP 10 MOVIES
By tickets sold:
      Movie_Title  Tickets_Sold  Ticket_Revenue
The Last Journey           854       263620.27
  Laugh Out Loud           724       215132.62
    Galaxy Quest           644       205118.02
    Ocean Planet           626       199794.84
    Robot Reborn           585       172945.33
     Hidden City           580       176875.24

By ticket revenue:
      Movie_Title  Tickets_Sold  Ticket_Revenue
The Last Journey           854       263620.27
  Laugh Out Loud           724       215132.62
    Galaxy Quest           644       205118.02
    Ocean Planet           626       199794.84
     Hidden City           580       176875.24
    Robot Reborn           585       172945.33


In [8]:
header("7. SCREEN FORMAT SUMMARY")
fmt_summary = (df.groupby(COL["fmt"])
                 .agg(Avg_Occupancy=(COL["occ"], "mean"),
                      Avg_Ticket_Price=(COL["price"], "mean"),
                      Tickets_Sold=(COL["qty"], "sum"),
                      Ticket_Revenue=("Ticket_Revenue", "sum"),
                      Avg_Concession=(COL["conc"], "mean"))
                 .sort_values("Avg_Occupancy", ascending=False))
print(fmt_summary.round(2))
print("\nFull numeric means (groupby.mean):")
print(df.groupby(COL["fmt"]).mean(numeric_only=True).round(2))


7. SCREEN FORMAT SUMMARY
               Avg_Occupancy  Avg_Ticket_Price  Tickets_Sold  Ticket_Revenue  Avg_Concession
Screen_Format                                                                               
Standard 2D            61.61            306.95          1391       425058.67          335.36
IMAX                   61.31            303.79          1359       415024.40          330.27
3D                     61.23            309.81          1263       393403.25          356.97

Full numeric means (groupby.mean):
               Tickets_Quantity  Ticket_Price  Occupancy_Percentage  Concession_Spend  Conc_Raw  Ticket_Revenue  Total_Spend
Screen_Format                                                                                                               
3D                         3.91        309.81                 61.23            356.97    358.08         1217.97      1574.93
IMAX                       4.02        303.79                 61.31            330.27    330.53   

In [9]:
header("8. WEEKDAY vs WEEKEND")
dt = (df.groupby("Day_Type")
        .agg(Transactions=(COL["txn"], "count"),
             Tickets_Sold=(COL["qty"], "sum"),
             Avg_Tickets_per_Txn=(COL["qty"], "mean"),
             Ticket_Revenue=("Ticket_Revenue", "sum"),
             Concession_Total=(COL["conc"], "sum"),
             Avg_Concession=(COL["conc"], "mean")))
print(dt.round(2))
# Per-day averages so 5 weekdays vs 2 weekend days are comparable
n_days = df.groupby("Day_Type")[COL["date"]].nunique()
print("\nPer calendar day:")
print((dt[["Tickets_Sold", "Ticket_Revenue", "Concession_Total"]]
       .div(n_days, axis=0)).round(2))


8. WEEKDAY vs WEEKEND
          Transactions  Tickets_Sold  Avg_Tickets_per_Txn  Ticket_Revenue  Concession_Total  Avg_Concession
Day_Type                                                                                                   
Weekday            703          2837                 4.04       884580.61         237385.08          337.67
Weekend            287          1176                 4.10       348905.71          99881.44          348.02

Per calendar day:
          Tickets_Sold  Ticket_Revenue  Concession_Total
Day_Type                                                
Weekday           7.79         2430.17            652.16
Weekend           7.95         2357.47            674.87


In [10]:
header("9. SHOWTIME SLOT ATTENDANCE")
slots = (df.groupby(COL["slot"])
           .agg(Transactions=(COL["txn"], "count"),
                Tickets_Sold=(COL["qty"], "sum"),
                Avg_Occupancy=(COL["occ"], "mean"),
                Avg_Concession=(COL["conc"], "mean"))
           .sort_values("Tickets_Sold", ascending=False))
slots["Ticket_Share_%"] = slots["Tickets_Sold"] / slots["Tickets_Sold"].sum() * 100
print(slots.round(2))


9. SHOWTIME SLOT ATTENDANCE
               Transactions  Tickets_Sold  Avg_Occupancy  Avg_Concession  Ticket_Share_%
Showtime_Slot                                                                           
Matinee                 338          1381          63.31          331.91           34.41
Evening                 325          1323          59.72          347.06           32.97
Late Night              327          1309          61.05          343.39           32.62


In [11]:
header("10. CONCESSION ATTACH RATE")
# Measured on ORIGINAL data, before imputation: a blank spend is treated as "no purchase".
# (After imputation every row would have spend > 0 and the rate would be a meaningless 100%.)
df["Has_Concession"] = df["Conc_Raw"].fillna(0) > 0
attach = df["Has_Concession"].mean() * 100
known = df["Conc_Raw"].dropna()
print(f"Attach rate (blank = no purchase): {attach:.1f}%")
print(f"Attach rate among rows with recorded spend: {(known > 0).mean() * 100:.1f}% "
      f"({len(known)} rows; zero-spend rows: {(known == 0).sum()})")
print("\nBy day type (%):\n", (df.groupby("Day_Type")["Has_Concession"].mean() * 100).round(1))
print("\nBy slot (%):\n", (df.groupby(COL["slot"])["Has_Concession"].mean() * 100).round(1))
print("\nBy genre (%):\n", (df.groupby(COL["genre"])["Has_Concession"].mean() * 100)
      .sort_values(ascending=False).round(1))


10. CONCESSION ATTACH RATE
Attach rate (blank = no purchase): 97.0%
Attach rate among rows with recorded spend: 100.0% (960 rows; zero-spend rows: 0)

By day type (%):
 Day_Type
Weekday    96.4
Weekend    98.3
Name: Has_Concession, dtype: float64

By slot (%):
 Showtime_Slot
Evening       96.6
Late Night    95.7
Matinee       98.5
Name: Has_Concession, dtype: float64

By genre (%):
 Genre
Sci-Fi       98.5
Comedy       98.1
Animation    97.8
Drama        96.3
Action       93.8
Name: Has_Concession, dtype: float64


In [12]:
header("11. GENRE: AVG TICKET vs SNACK SPEND")
genre = (df.groupby(COL["genre"])
           .agg(Avg_Ticket_Revenue=("Ticket_Revenue", "mean"),
                Avg_Ticket_Price=(COL["price"], "mean"),
                Avg_Snack_Spend=(COL["conc"], "mean"),
                Transactions=(COL["txn"], "count")))
genre["Snack_to_Ticket_Ratio"] = genre["Avg_Snack_Spend"] / genre["Avg_Ticket_Revenue"]
genre = genre.sort_values("Avg_Snack_Spend", ascending=False)
print(genre.round(2))


11. GENRE: AVG TICKET vs SNACK SPEND
           Avg_Ticket_Revenue  Avg_Ticket_Price  Avg_Snack_Spend  Transactions  Snack_to_Ticket_Ratio
Genre                                                                                                
Animation             1239.00            295.12           347.97           182                   0.28
Sci-Fi                1244.30            312.22           345.47           203                   0.28
Drama                 1169.22            310.34           343.03           219                   0.29
Comedy                1279.61            312.50           337.42           208                   0.26
Action                1309.98            301.58           328.63           178                   0.25


In [13]:
header("12. NUMPY STATS: TOTAL TRANSACTION SPEND")
spend = df["Total_Spend"].to_numpy(dtype=float)
print(f"Mean   : {np.mean(spend):.2f}")
print(f"Median : {np.median(spend):.2f}")
print(f"Std dev: {np.std(spend, ddof=1):.2f} (sample)")


12. NUMPY STATS: TOTAL TRANSACTION SPEND
Mean   : 1586.62
Median : 1504.52
Std dev: 760.79 (sample)


In [14]:
header("RECOMMENDATIONS (generated from the results above)")
fmt_gap = fmt_summary["Avg_Occupancy"].max() - fmt_summary["Avg_Occupancy"].min()
best_fmt = fmt_summary["Avg_Occupancy"].idxmax()
worst_fmt = fmt_summary["Avg_Occupancy"].idxmin()
we_share = dt.loc["Weekend", "Tickets_Sold"] / dt["Tickets_Sold"].sum() * 100
slot_attach = df.groupby(COL["slot"])["Has_Concession"].mean() * 100
low_slot = slots["Avg_Occupancy"].idxmin()
top_slot = slots["Tickets_Sold"].idxmax()
slot_gap = slots["Avg_Occupancy"].max() - slots["Avg_Occupancy"].min()
high_snack_genre = genre["Avg_Snack_Spend"].idxmax()
low_snack_genre = genre["Avg_Snack_Spend"].idxmin()
top_movie = top_by_tickets.iloc[0][COL["movie"]]
top_share = top_by_tickets.iloc[0]["Tickets_Sold"] / movies["Tickets_Sold"].sum() * 100
wd_conc = dt.loc["Weekday", "Avg_Concession"]; we_conc = dt.loc["Weekend", "Avg_Concession"]

recs = [
    f"1. Screen allocation: average occupancy differs by only {fmt_gap:.1f} points across formats "
    f"({best_fmt} {fmt_summary.loc[best_fmt, 'Avg_Occupancy']:.1f}% vs {worst_fmt} "
    f"{fmt_summary.loc[worst_fmt, 'Avg_Occupancy']:.1f}%), so format alone does not justify cutting screens. "
    f"Allocate by title demand instead: '{top_movie}' sells {top_share:.0f}% of all tickets and should get the most screens/showtimes.",
    f"2. Weekend scheduling: weekends carry {we_share:.0f}% of tickets from only 2 of 7 days, and per-day ticket "
    f"volume is about the same as weekdays. Weekend snack spend per transaction is {we_conc:.0f} vs {wd_conc:.0f} "
    f"on weekdays, so keep weekday pricing/promotions to lift weekday seats and staff the snack bar for weekend peaks.",
    f"3. Showtime slots: {top_slot} draws the most tickets, while {low_slot} has the lowest occupancy "
    f"({slots.loc[low_slot, 'Avg_Occupancy']:.1f}%; slot spread {slot_gap:.1f} points). Consider discounted "
    f"tickets or themed programming in {low_slot} rather than cutting it.",
    f"4. Combo promotions: attach rate is {attach:.1f}% (blank spend = no purchase), so the opportunity is "
    f"raising spend per transaction, not attach rate. Offer tiered combos (ticket + snack bundles) and upsells "
    f"at checkout; weakest attach is {slot_attach.idxmin()} ({slot_attach.min():.1f}%).",
    f"5. Genre pairing: {high_snack_genre} has the highest average snack spend "
    f"({genre.loc[high_snack_genre, 'Avg_Snack_Spend']:.0f}); lead premium combos there. "
    f"{low_snack_genre} is lowest ({genre.loc[low_snack_genre, 'Avg_Snack_Spend']:.0f}), "
    f"so test pre-show snack offers on those screenings.",
]
for r in recs:
    print(r, "\n")


RECOMMENDATIONS (generated from the results above)
1. Screen allocation: average occupancy differs by only 0.4 points across formats (Standard 2D 61.6% vs 3D 61.2%), so format alone does not justify cutting screens. Allocate by title demand instead: 'The Last Journey' sells 21% of all tickets and should get the most screens/showtimes. 

2. Weekend scheduling: weekends carry 29% of tickets from only 2 of 7 days, and per-day ticket volume is about the same as weekdays. Weekend snack spend per transaction is 348 vs 338 on weekdays, so keep weekday pricing/promotions to lift weekday seats and staff the snack bar for weekend peaks. 

3. Showtime slots: Matinee draws the most tickets, while Evening has the lowest occupancy (59.7%; slot spread 3.6 points). Consider discounted tickets or themed programming in Evening rather than cutting it. 

4. Combo promotions: attach rate is 97.0% (blank spend = no purchase), so the opportunity is raising spend per transaction, not attach rate. Offer tiere

In [15]:
df.to_csv("cinema_sales_cleaned.csv", index=False)
summaries = {"top10_tickets": top_by_tickets, "top10_revenue": top_by_revenue,
             "screen_format": fmt_summary, "weekday_vs_weekend": dt,
             "showtime_slots": slots, "genre": genre}
for name, tbl in summaries.items():
    tbl.to_csv(f"summary_{name}.csv", index=name not in ("top10_tickets", "top10_revenue"))
print("Saved: cinema_sales_cleaned.csv and summary_*.csv tables")

Saved: cinema_sales_cleaned.csv and summary_*.csv tables
